In [ ]:
import glob
import seaborn as sns
import pandas as pd

In [ ]:
from utils.files import load_json
from os.path import join, getctime


def get_latest_ae_results(exp_path, checkpoint=100000):
    results_paths = glob.glob(join(exp_path, 'eval', 'Hybrid-Autencoder', str(checkpoint), 'results.json'), recursive=True)
    return load_json(max(results_paths, key=getctime))

def get_ae_results(exp_path, checkpoint=100000):
    return pd.read_csv(join(exp_path, 'policy_actions', str(checkpoint), 'ae_losses.csv'))


In [ ]:
experiments = [
    
    
    'logs/e2e-repeat-V-CQL-2025-07-29 20:46:42.095383/0/V-CQL-2025-07-29 20:46:41.233309',
    'logs/e2e-repeat-V-CQL-2025-07-31 21:47:53.272687/1/V-CQL-2025-07-31 21:47:52.489954',
    'logs/e2e-repeat-CQL-2025-07-28 19:59:01.108261/0/CQL-2025-07-28 19:59:00.274045',
    'logs/e2e-F-CQL-2025-10-06 17:00:50.875076/F-CQL-2025-10-06 17:00:50.333008',
    'logs/e2e-repeat-factored-CQL-2025-07-29 10:40:01.758449/1/factored-CQL-2025-07-29 10:40:00.914841',
    'logs/e2e-Hybrid-IQL-2025-10-02 16:19:35.502138/Hybrid-IQL-2025-10-02 16:19:34.830469',
    'logs/e2e-repeat-Hybrid-EDAC-2025-07-29 00:47:08.777956/0/Hybrid-EDAC-2025-07-29 00:47:07.972207'
]

In [ ]:
names = {
    'CQL': 'C-CQL',
    'factored-CQL': 'CF-CQL',
    'V-CQL': 'CQL',
    'Hybrid-IQL': 'HybridIQL',
    'Hybrid-EDAC': 'HybridEDAC',
    'Discrete-IQL': 'DiscreteIQL',

}
results = []
for exp_path in experiments:
    config = load_json(join(exp_path, 'config.json'))
    algo_name = config['name']
        
        
    print(exp_path)
    ae_loss = get_ae_results(exp_path)
    
    algo_name = names[algo_name] if algo_name in names else algo_name
    
    if algo_name == 'CQL' and config["alpha"]>0.1:
        ae_loss['algo_name'] = f'{algo_name} (α={config["alpha"]})'
    else:
        ae_loss['algo_name'] = algo_name
   
    
    results.append(ae_loss)


results = pd.concat(results)
results.sort_values(by=['loss'], inplace=True)
#results['loss'] = -results['loss']
print(results)

In [ ]:
import numpy as np
from dataset.pre_processing_configs import load_pre_processing_configs
from dataset.config import load_dataset_config
from torch import tensor
from agents.base import Agent
from algorithms.eval.eval_ood_hybrid_ae_dataset_policy import eval_get_all_losses
from actions.continuous import disc_to_cont_using_mode
from actions.discrete_actions import get_bins_per_action_dim
import torch
from actions.hybrid import get_continuous_action, get_discrete_action, create_hybrid_action_tensor_dict
from dataset.pre_processing import un_normalize_data, normalize_data
from actions.space import get_discrete_actions_list, get_continuous_actions_list, is_only_discrete
from dataset.load import load_dataset_to_buffer
from os.path import getmtime
from dotenv import load_dotenv
from os import getenv
load_dotenv()
ae_experiment_path = getenv('AE_EXPERIMENT_PATH')
ae_checkpoints_path = join(ae_experiment_path, 'checkpoints')
ae_checkpoints = glob.glob(f"{ae_checkpoints_path}/*.pkl")
ae_checkpoints.sort(key=getmtime, reverse=True)
ae_checkpoint = ae_checkpoints[0]

def get_dataset_ae_loss(dataset_config, pre_processing_configs, ae_dataset_config, device, ae_checkpoint,
                        ae_pre_processing_config):
    buffer = load_dataset_to_buffer(dataset=dataset_config.test_dataset_split,
                                    dataset_configs=dataset_config,
                                    pre_process_configs=pre_processing_configs,
                                    device=device)
    policy_actions = buffer.actions

    discrete_actions_list = get_discrete_actions_list(action_space=pre_processing_configs.action_space)
    continuous_actions_list = get_continuous_actions_list(action_space=pre_processing_configs.action_space)

    continuous_actions_dict = {}
    buffer_states_df = pd.DataFrame(columns=pre_processing_configs.get_list_of_states(),
                                    data=buffer.observations.cpu().numpy())
    un_normalized_states = un_normalize_data(
        dataset=buffer_states_df,
        columns=pre_processing_configs.get_list_of_states(),
        norm_dict=dataset_config.normalization_params
    )
    ae_normalized_states = normalize_data(
        dataset=un_normalized_states,
        columns=pre_processing_configs.get_list_of_states(),
        norm_dict=ae_dataset_config.normalization_params
    )
    if continuous_actions_list:
        policy_actions_cont = get_continuous_action(policy_actions)
        policy_actions_cont = pd.DataFrame(columns=continuous_actions_list, data=policy_actions_cont.cpu().numpy())
        dataset_norm_dict = dataset_config.normalization_params
        un_normalized_policy_actions_cont = un_normalize_data(dataset=policy_actions_cont,
                                                              columns=continuous_actions_list,
                                                              norm_dict=dataset_norm_dict)

        ae_normalized_policy_actions_cont = normalize_data(dataset=un_normalized_policy_actions_cont,
                                                           columns=continuous_actions_list,
                                                           norm_dict=ae_dataset_config.normalization_params)
        ae_normalized_policy_actions_cont = ae_normalized_policy_actions_cont[continuous_actions_list].values
        ae_normalized_policy_actions_cont = torch.tensor(ae_normalized_policy_actions_cont, device=device)

        continuous_actions_dict = {
            key: ae_normalized_policy_actions_cont[:, i] for i, key in enumerate(continuous_actions_list)
        }

    bins_per_action_dim = get_bins_per_action_dim(
        actions_ranges=dataset_config.discrete_action_bin_ranges,
        list_of_actions=discrete_actions_list,
        vent_mode_conditional_null_bins=pre_processing_configs.vent_mode_action_masking
    )
    if is_only_discrete(pre_processing_configs.action_space):
        policy_actions_disc = policy_actions
    else:
        policy_actions_disc = get_discrete_action(actions=policy_actions)

    converted_actions = disc_to_cont_using_mode(
        actions=policy_actions_disc,
        list_of_actions=discrete_actions_list,
        bins_per_action_dim=bins_per_action_dim,
        discrete_action_bin_ranges=dataset_config.discrete_action_bin_ranges
    )

    ae_cont_actions_list = get_continuous_actions_list(action_space=ae_pre_processing_config.action_space)
    norm_var = list(set(discrete_actions_list).intersection(ae_cont_actions_list))
    converted_actions_red = {name: actions.cpu() for name, actions in converted_actions.items() if name in norm_var}
    if norm_var:
        converted_actions_normalized = normalize_data(
            dataset=pd.DataFrame(converted_actions_red),
            columns=norm_var,
            norm_dict=ae_dataset_config.normalization_params
        )
        converted_actions_normalized = converted_actions_normalized.to_dict('list')

        combined_actions_dict = {**converted_actions_normalized, **continuous_actions_dict}
    else:
        combined_actions_dict = continuous_actions_dict

    combined_cont_actions_list = [tensor(combined_actions_dict[action], device=device).reshape(-1, 1) for action in
                                  ae_cont_actions_list]
    combined_cont_actions = torch.cat(combined_cont_actions_list, dim=-1).to(device=device)

    buffer.actions = create_hybrid_action_tensor_dict(
        continuous_actions=combined_cont_actions,
        discrete_actions=converted_actions['vent_mode'].to(device=device)
    )

    print(buffer.actions['continuous_actions'].shape)
    buffer.observations = torch.tensor(ae_normalized_states[pre_processing_configs.get_list_of_states()].values,
                                       device=device)

    agent = Agent.load(save_path=ae_checkpoint)

    return eval_get_all_losses(agent=agent, eval_buffer=buffer)

dataset_config_path = join(ae_experiment_path, 'dataset_config.json')
dataset_config = load_dataset_config(dataset_config_path=dataset_config_path, experiment_path=ae_experiment_path)

ae_pre_processing_config_path = join(ae_experiment_path, 'pre_processing_metadata.json')
ae_pre_processing_config = load_pre_processing_configs(pre_process_configs_path=ae_pre_processing_config_path)


clinician_dpi = get_dataset_ae_loss(dataset_config=dataset_config, pre_processing_configs=ae_pre_processing_config, ae_dataset_config=dataset_config, device='cuda', ae_checkpoint=ae_checkpoint, ae_pre_processing_config=ae_pre_processing_config)
clinician_dpi = clinician_dpi['loss'].values
q1 = np.percentile(clinician_dpi, 25)
iqr = np.percentile(clinician_dpi, 75) - q1
ood_threshold = q1 - 1.5 * iqr

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# --- clean & set categorical order
order = ["CQL","CQL(alpha=0.5)","C-CQL","CF-CQL","HybridEDAC","HybridIQL"]
order = pd.Index(order).drop_duplicates().tolist()

df = results.copy()
df = df[df["algo_name"].notna()]  # avoid NaNs in category
df["algo_name"] = pd.Categorical(df["algo_name"], categories=order, ordered=True)

sns.set_theme(style="whitegrid", context="notebook")
fig, ax = plt.subplots(figsize=(8, 5))

ax = sns.boxplot(
    x="algo_name", y="loss", data=df, width=0.6, order=order,
    showfliers=True,
    flierprops=dict(marker='o', markersize=3, markerfacecolor='none',
                    markeredgecolor='0.3', alpha=0.35),
    boxprops=dict(linewidth=1.0),
    whiskerprops=dict(linewidth=1.0),
    capprops=dict(linewidth=1.0),
    medianprops=dict(linewidth=1.4, color='black')
)

# semi-transparent boxes
for patch in ax.artists:
    patch.set_alpha(0.6)

# OOD line + legend
ood_line = ax.axhline(ood_threshold, color='red', linestyle='--', linewidth=1.5)
legend_line = Line2D([0], [0], color='red', linestyle='--', linewidth=1.5, label='OOD threshold')
ax.legend(handles=[legend_line], loc='lower right', frameon=True, facecolor='white', edgecolor='0.7')

ax.set_xlabel("Algorithm", fontsize=12)
ax.set_ylabel(r"$d^\pi$", fontsize=12)
# let seaborn set tick labels from the categorical order; just rotate if you like
for tick in ax.get_xticklabels():
    tick.set_rotation(0)

# keep OOD line visible
ymin = min(df["loss"].min(), ood_threshold) - 0.5
ymax = max(df["loss"].max(), ood_threshold) + 0.5
ax.set_ylim(ymin, ymax)

plt.tight_layout()
plt.show()

In [ ]:
results[['algo_name', 'loss']]